# Multi-agent Customer Support Automation

In this notebook, you will learn about the six key elements which help make Agents perform even better:
- Role Playing
- Focus
- Tools
- Cooperation
- Guardrails
- Memory

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, API and LLM

In [3]:
from crewai import Agent, Task, Crew

In [4]:
from dotenv import load_dotenv
import os

load_dotenv()

True

## Role Playing, Focus and Cooperation

In [5]:
support_agent = Agent(
    role="Senior Support Representative",
	goal="Be the most friendly and helpful "
        "support representative in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        " are now working on providing "
		"support to {customer}, a super important customer "
        " for your company."
		"You need to make sure that you provide the best support!"
		"Make sure to provide full complete answers, "
        " and make no assumptions."
	),
	allow_delegation=False,
	verbose=True
)

- By not setting `allow_delegation=False`, `allow_delegation` takes its default value of being `True`.
- This means the agent _can_ delegate its work to another agent which is better suited to do a particular task. 

In [6]:
support_quality_assurance_agent = Agent(
	role="Support Quality Assurance Specialist",
	goal="Get recognition for providing the "
    "best support quality assurance in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        "are now working with your team "
		"on a request from {customer} ensuring that "
        "the support representative is "
		"providing the best support possible.\n"
		"You need to make sure that the support representative "
        "is providing full"
		"complete answers, and make no assumptions."
	),
	verbose=True
)

* **Role Playing**: Both agents have been given a role, goal and backstory.
* **Focus**: Both agents have been prompted to get into the character of the roles they are playing.
* **Cooperation**: Support Quality Assurance Agent can delegate work back to the Support Agent, allowing for these agents to work together.

## Tools, Guardrails and Memory

### Tools

- Import CrewAI tools

```pip install crewai-tools```

In [7]:
from crewai_tools import SerperDevTool, \
                         ScrapeWebsiteTool, \
                         WebsiteSearchTool

### Possible Custom Tools
- Load customer data
- Tap into previous conversations
- Load data from a CRM
- Checking existing bug reports
- Checking existing feature requests
- Checking ongoing tickets
- ... and more

- Some ways of using CrewAI tools.

```Python
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
```

- Instantiate a document scraper tool.
- The tool will scrape a page (only 1 URL) of the CrewAI documentation.

In [8]:
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/en/concepts/crews"
)

##### Different Ways to Give Agents Tools

- Agent Level: The Agent can use the Tool(s) on any Task it performs.
- Task Level: The Agent will only use the Tool(s) when performing that specific Task.

**Note**: Task Tools override the Agent Tools.

### Creating Tasks
- You are passing the Tool on the Task Level.

In [9]:
inquiry_resolution = Task(
    description=(
        "{customer} just reached out with a super important ask:\n"
	    "{inquiry}\n\n"
        "{person} from {customer} is the one that reached out. "
		"Make sure to use everything you know "
        "to provide the best support possible."
		"You must strive to provide a complete "
        "and accurate response to the customer's inquiry."
    ),
    expected_output=(
	    "A detailed, informative response to the "
        "customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include references "
        "to everything you used to find the answer, "
        "including external data or solutions. "
        "Ensure the answer is complete, "
		"leaving no questions unanswered, and maintain a helpful and friendly "
		"tone throughout."
    ),
	tools=[docs_scrape_tool],
    agent=support_agent,
)

- `quality_assurance_review` is not using any Tool(s)
- Here the QA Agent will only review the work of the Support Agent

In [10]:
quality_assurance_review = Task(
    description=(
        "Review the response drafted by the Senior Support Representative for {customer}'s inquiry. "
        "Ensure that the answer is comprehensive, accurate, and adheres to the "
		"high-quality standards expected for customer support.\n"
        "Verify that all parts of the customer's inquiry "
        "have been addressed "
		"thoroughly, with a helpful and friendly tone.\n"
        "Check for references and sources used to "
        " find the information, "
		"ensuring the response is well-supported and "
        "leaves no questions unanswered."
    ),
    expected_output=(
        "A final, detailed, and informative response "
        "ready to be sent to the customer.\n"
        "This response should fully address the "
        "customer's inquiry, incorporating all "
		"relevant feedback and improvements.\n"
		"Don't be too formal, we are a chill and cool company "
	    "but maintain a professional and friendly tone throughout."
    ),
    agent=support_quality_assurance_agent,
)


### Creating the Crew

#### Memory
- Setting `memory=True` when putting the crew together enables Memory.

In [11]:
crew = Crew(
  agents=[support_agent, support_quality_assurance_agent],
  tasks=[inquiry_resolution, quality_assurance_review],
  verbose=True,
  memory=True
)

### Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

#### Guardrails
- By running the execution below, you can see that the agents and the responses are within the scope of what we expect from them.

In [13]:
inputs = {
    "customer": "Class.vision",
    "person": "Alireza Akhavanpour",
    "inquiry": "I need help with setting up a Crew "
               "and kicking it off, specifically "
               "how can I add memory to my crew? "
               "Can you provide guidance?"
}
result = crew.kickoff(inputs=inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 383d1bd7-4a02-4039-8ac5-87c5a9aa0196                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Class.vision just reached out with a super important ask:                                                │
│  I need help with setting up a Crew and kicking it off, specifically how can I add memory to my crew? Can you   │
│  provide guidance?                                                                                              │
│                                                                                                                 │
│  Alireza Akhavanpour from Class.vision is the one that reached out. Make sure to use everything you know to     │
│  provide the best support possible.You must strive to provide a complete and accurate response to the           │
│  customer's inquiry.                                                                                            │
│  ID: 92938f96-2799-4396-97f8-56410b0859cf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Support Representative                                                                           │
│                                                                                                                 │
│  Task: Class.vision just reached out with a super important ask:                                                │
│  I need help with setting up a Crew and kicking it off, specifically how can I add memory to my crew? Can you   │
│  provide guidance?                                                                                              │
│                                                                                                                 │
│  Alireza Akhavanpour from Class.vision is the one that reached out. Make sure to use everything you know to     │
│  provide the best support possible.You must strive to provide a complete and accurate response to the           │
│  customer's inquiry.                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: read_website_content                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool read_website_content executed with result: The following text is scraped website content:

Crews - CrewAI Skip to main content CrewAI home page v1.14.2 English Search... ⌘ K Start Cloud Trial crewAIInc/crewAI crewAIInc/crewAI Search... Navigat...


...

Tool search_memory executed with result: Found memories:
- (score=0.85) Memory functionality in CrewAI allows the Crew to retain context and improve over time.
  categories: memory, artificial intelligence, features
  entities: []
  dates: [...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Assurance Specialist                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hello Alireza,                                                                                                 │
│                                                                                                                 │
│  Thank you for reaching out with your question about setting up a Crew and adding memory in CrewAI. I’m here    │
│  to guide you through the process so you can get your Crew up and running smoothly.                             │
│                                                                                                                 │
│  In CrewAI, a Crew represents a collaborative group of agents working together to complete tasks. One powerful  │
│  feature is the ability to add memory to your Crew. This memory stores execution-related information—such as    │
│  short-term, long-term, and entity memories—to help your Crew retain context, improve decision-making, and      │
│  perform better over time.                                                                                      │
│                                                                                                                 │
│  Here’s how you can add memory to your Crew:                                                                    │
│                                                                                                                 │
│  1. When you create your Crew object, include the memory attribute. This attribute can be configured to use     │
│  different types of memories depending on your needs.                                                           │
│                                                                                                                 │
│  2. The memory configuration is passed as a parameter during the Crew initialization. Here's an example in      │
│  Python:                                                                                                        │
│                                                                                                                 │
│  ```python                                                                                                      │
│  from crewai import Crew, Agent, Task, Process                                                                  │
│                                                                                                                 │
│  # Define your agents and tasks                                                                                 │
│  agents = [agent1, agent2]                                                                                      │
│  tasks = [task1, task2]                                                                                         │
│                                                                                                                 │
│  # Define your memory configuration (customize as needed)                                                       │
│  memory_config = {                                                                                              │
│      "type": "short_term",  # Or "long_term", "entity", or your specific memory implementation                  │
│      "parameters": {                                                                                            │
│          # Insert your memory-specific parameters here 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the response drafted by the Senior Support Representative for Class.vision's inquiry. Ensure      │
│  that the answer is comprehensive, accurate, and adheres to the high-quality standards expected for customer    │
│  support.                                                                                                       │
│  Verify that all parts of the customer's inquiry have been addressed thoroughly, with a helpful and friendly    │
│  tone.                                                                                                          │
│  Check for references and sources used to  find the information, ensuring the response is well-supported and    │
│  leaves no questions unanswered.                                                                                │
│  Agent: Support Quality Assurance Specialist                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 383d1bd7-4a02-4039-8ac5-87c5a9aa0196                                                                       │
│  Final Output: Hello Alireza,                                                                                   │
│                                                                                                                 │
│  Thank you for reaching out with your question about setting up a Crew and adding memory in CrewAI. I’m here    │
│  to guide you through the process so you can get your Crew up and running smoothly.                             │
│                                                                                                                 │
│  In CrewAI, a Crew represents a collaborative group of agents working together to complete tasks. One powerful  │
│  feature is the ability to add memory to your Crew. This memory stores execution-related information—such as    │
│  short-term, long-term, and entity memories—to help your Crew retain context, improve decision-making, and      │
│  perform better over time.                                                                                      │
│                                                                                                                 │
│  Here’s how you can add memory to your Crew:                                                                    │
│                                                                                                                 │
│  1. When you create your Crew object, include the memory attribute. This attribute can be configured to use     │
│  different types of memories depending on your needs.                                                           │
│                                                                                                                 │
│  2. The memory configuration is passed as a parameter during the Crew initialization. Here's an example in      │
│  Python:                                                                                                        │
│                                                                                                                 │
│  ```python                                                                                                      │
│  from crewai import Crew, Agent, Task, Process                                                                  │
│                                                                                                                 │
│  # Define your agents and tasks                                                                                 │
│  agents = [agent1, agent2]                                                                                      │
│  tasks = [task1, task2]                                                                                         │
│                                                                                                                 │
│  # Define your memory configuration (customize as needed)                                                       │
│  memory_config = {                                                                                              │
│      "type": "short_term",  # Or "long_term", "entity", or your specific memory implementation                  │
│      "parameters": {                                                                                            │
│          # Insert your memory-specific parameters here

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the final result as Markdown.

In [14]:
from IPython.display import Markdown
Markdown(result.raw)

Hello Alireza,

Thank you for reaching out with your question about setting up a Crew and adding memory in CrewAI. I’m here to guide you through the process so you can get your Crew up and running smoothly.

In CrewAI, a Crew represents a collaborative group of agents working together to complete tasks. One powerful feature is the ability to add memory to your Crew. This memory stores execution-related information—such as short-term, long-term, and entity memories—to help your Crew retain context, improve decision-making, and perform better over time.

Here’s how you can add memory to your Crew:

1. When you create your Crew object, include the memory attribute. This attribute can be configured to use different types of memories depending on your needs.

2. The memory configuration is passed as a parameter during the Crew initialization. Here's an example in Python:

```python
from crewai import Crew, Agent, Task, Process

# Define your agents and tasks
agents = [agent1, agent2]
tasks = [task1, task2]

# Define your memory configuration (customize as needed)
memory_config = {
    "type": "short_term",  # Or "long_term", "entity", or your specific memory implementation
    "parameters": {
        # Insert your memory-specific parameters here
    }
}

# Create the Crew and include memory
crew = Crew(
    agents=agents,
    tasks=tasks,
    process=Process.sequential,
    memory=memory_config,
    verbose=True
)

# Kick off the Crew with input data
result = crew.kickoff(inputs={"your_input_key": "your_input_value"})
print(result.raw)
```

3. If you prefer YAML configuration, you can similarly specify the memory section under your Crew setup to define and customize memory behavior.

4. The memory functionality helps the Crew retain context between runs and learn over time, which is invaluable for complex workflows or recurring tasks.

Additionally, CrewAI offers features like checkpointing and caching, which you can enable alongside memory to make your Crew more resilient and efficient.

If you'd like, I can provide more detailed examples or help you configure advanced memory types tailored to your use cases—just let me know!

Looking forward to helping you make the most of CrewAI.

Best regards,  
[Your Name]  
Senior Support Representative  
crewAI